# Foreign Whispers — GPU Services on Colab

This notebook runs **Whisper STT** and **Chatterbox TTS** on a Colab GPU,
then exposes them via **ngrok** tunnels so your local Docker Compose can use them.

**Requirements:**
- Colab runtime with GPU (T4, A100, etc.)
- Free ngrok account — get your auth token at https://dashboard.ngrok.com/get-started/your-authtoken

**How it works:**
- Uses `faster-whisper` (the engine behind speaches) for STT
- Uses `chatterbox-tts` (Resemble AI) for TTS
- Wraps both in minimal FastAPI servers matching the OpenAI-compatible API
- Exposes them via ngrok tunnels for your local Docker API to connect to

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
# Install uv for fast, reliable dependency resolution
!pip install -q uv

# faster-whisper + API server deps (no torch conflicts)
!uv pip install --system -q faster-whisper pyngrok fastapi uvicorn python-multipart soundfile

# Colab ships torch 2.10+, numpy 2.x, and torchaudio pre-installed with CUDA.
# chatterbox-tts and resemble-enhance pin older torch/numpy versions that
# break Colab's C extensions. The override file tells uv to keep Colab's
# versions instead of downgrading.
import pathlib
pathlib.Path("/tmp/overrides.txt").write_text(
    "torch>=2.0\nnumpy>=1.24\ntorchaudio>=2.0\n"
)

# Resolve ALL chatterbox deps properly — overrides prevent torch/numpy downgrade,
# --prerelease=allow picks up resemble-enhance 0.0.2.dev (only build for py3.12)
!uv pip install --system -q --prerelease=allow --override /tmp/overrides.txt chatterbox-tts

print("All dependencies installed.")

## 3. Set your ngrok auth token

Get it from https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_AUTH_TOKEN = ""  # <-- paste your ngrok auth token here

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok authenticated.")

## 4. Load models

First run downloads weights (~3 GB total). Subsequent runs use the cache.

In [ ]:
import torch, torchaudio, time, warnings

# Suppress HF Hub auth warning — Whisper models are public, no token needed
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")

# ── Whisper STT ──────────────────────────────────────────────
print("Loading Whisper medium model...")
from faster_whisper import WhisperModel
whisper_model = WhisperModel("medium", device="cuda", compute_type="float16")
print("  Whisper ready.")

# ── Chatterbox TTS ───────────────────────────────────────────
print("Loading Chatterbox TTS model...")
from chatterbox.tts import ChatterboxTTS
chatterbox_model = ChatterboxTTS.from_pretrained(device="cuda")
SAMPLE_RATE = chatterbox_model.sr
print(f"  Chatterbox ready (sample rate: {SAMPLE_RATE}).")

## 5. Start API servers

Creates two FastAPI apps that match the OpenAI-compatible endpoints
the Foreign Whispers API expects, then runs them in background threads.

In [ ]:
import io, os, subprocess, tempfile, threading
from fastapi import FastAPI, File, Form, UploadFile
from fastapi.responses import Response
import uvicorn
import soundfile as sf

# Kill any previous server occupying port 8000
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(1)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Combined GPU services server (port 8000)
# ngrok free tier only allows ONE tunnel, so both Whisper and
# Chatterbox share a single FastAPI app. Their paths don't conflict:
#   Whisper:    POST /v1/audio/transcriptions
#   Chatterbox: POST /v1/audio/speech, POST /v1/audio/speech/upload
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
app = FastAPI(title="Foreign Whispers GPU Services")

@app.get("/health")
def health():
    return {"status": "ok", "services": ["whisper-stt", "chatterbox-tts"]}

# ── Whisper STT endpoint ─────────────────────────────────────
@app.post("/v1/audio/transcriptions")
async def transcriptions(
    file: UploadFile = File(...),
    response_format: str = Form("verbose_json"),
):
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as tmp:
        tmp.write(await file.read())
        tmp.flush()
        segments_gen, info = whisper_model.transcribe(tmp.name)
        segments = []
        full_text = []
        for seg in segments_gen:
            segments.append({
                "id": seg.id,
                "start": round(seg.start, 3),
                "end": round(seg.end, 3),
                "text": seg.text,
            })
            full_text.append(seg.text.strip())
    return {
        "language": info.language,
        "text": " ".join(full_text),
        "segments": segments,
        "duration": round(info.duration, 3),
    }

# ── Chatterbox TTS helpers ───────────────────────────────────
def _generate_wav_bytes(wav_tensor: torch.Tensor) -> bytes:
    """Convert model output tensor to WAV bytes.

    Uses soundfile (libsndfile) instead of torchaudio.save() because
    torchaudio 2.10+ (TorchCodec backend) cannot write to BytesIO.
    """
    if wav_tensor.is_cuda:
        wav_tensor = wav_tensor.cpu()
    if wav_tensor.dim() == 1:
        wav_tensor = wav_tensor.unsqueeze(0)
    data = wav_tensor.numpy().T
    buf = io.BytesIO()
    sf.write(buf, data, SAMPLE_RATE, format="WAV", subtype="PCM_16")
    return buf.getvalue()

# ── Chatterbox TTS endpoints ─────────────────────────────────
@app.post("/v1/audio/speech")
async def speech(body: dict):
    text = body.get("input", "")
    wav = chatterbox_model.generate(text)
    return Response(content=_generate_wav_bytes(wav), media_type="audio/wav")

@app.post("/v1/audio/speech/upload")
async def speech_upload(
    input: str = Form(...),
    response_format: str = Form("wav"),
    voice_file: UploadFile = File(...),
):
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp.write(await voice_file.read())
        tmp.flush()
        ref_path = tmp.name
    wav = chatterbox_model.generate(input, audio_prompt_path=ref_path)
    os.unlink(ref_path)
    return Response(content=_generate_wav_bytes(wav), media_type="audio/wav")

# ── Start server ─────────────────────────────────────────────
threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": 8000, "log_level": "warning"},
    daemon=True,
).start()

time.sleep(3)
print("GPU services running on port 8000 (Whisper STT + Chatterbox TTS)")

## 6. Create ngrok tunnels

In [ ]:
import subprocess, time as _time

# Force-kill ALL ngrok processes (pyngrok + orphans from previous runs)
subprocess.run(["pkill", "-9", "-f", "ngrok"], capture_output=True)
_time.sleep(5)  # Wait for ngrok edge servers to release the endpoint

tunnel = ngrok.connect(8000, "http")
gpu_url = tunnel.public_url

whisper_url = gpu_url
tts_url = gpu_url

print("=" * 60)
print("COPY THESE INTO YOUR .env FILE:")
print("=" * 60)
print()
print(f"FW_WHISPER_API_URL={gpu_url}")
print(f"CHATTERBOX_API_URL={gpu_url}")
print()
print("Then restart the API container:")
print("  docker compose --profile cpu up -d api")
print("=" * 60)

## 7. Health check

In [ ]:
import requests

# Test Whisper
try:
    r = requests.get(f"{whisper_url}/health", timeout=10)
    print(f"Whisper STT: {r.status_code} {r.json()}")
except Exception as e:
    print(f"Whisper STT: not ready yet ({e}). Wait a bit and re-run this cell.")

# Test Chatterbox
try:
    r = requests.get(f"{tts_url}/health", timeout=10)
    print(f"Chatterbox TTS: {r.status_code} {r.json()}")
except Exception as e:
    print(f"Chatterbox TTS: not ready yet ({e}). Wait a bit and re-run this cell.")

## 8. Quick TTS test

In [ ]:
import requests
from IPython.display import Audio

resp = requests.post(
    f"{tts_url}/v1/audio/speech",
    json={"input": "Hola, esta es una prueba del sistema de texto a voz.", "response_format": "wav"},
    timeout=60,
)
resp.raise_for_status()

with open("/tmp/test_tts.wav", "wb") as f:
    f.write(resp.content)

print(f"Generated {len(resp.content)} bytes of audio")
Audio("/tmp/test_tts.wav")

## 9. Keep alive

Run this cell to keep the notebook alive. The tunnels stay open as long as this cell is running.
Press the stop button when you're done.

In [ ]:
import time

print("Keeping services alive. Press stop to shut down.")
print(f"GPU services: {gpu_url}")
print()

try:
    while True:
        time.sleep(60)
        try:
            requests.get(f"{gpu_url}/health", timeout=5)
            print(".", end="", flush=True)
        except:
            print("x", end="", flush=True)
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.kill()
    print("Done.")